# 03_analysis
Goal: answer the main question with the gold tables: **where do we lose our customers, before they buy or after their first order?**
- **Input:** the gold tables in `dev.ecommerce` (built in `02_gold`)
- **How to read:** each finding has its queries (with the result), then a short note: what it says, the numbers, and what to be careful about. Finding 7 answers the main question.

In [0]:
%sql
USE CATALOG dev;
USE SCHEMA ecommerce

In [0]:
%sql
SELECT * FROM dev.ecommerce.gold_overview

In [0]:
%sql
with monthly as (
select
month,
buyers,
lag(buyers) over(order by month) as prev_buyers
from dev.ecommerce.gold_overview
)

select 
month,
buyers,
prev_buyers,
round((buyers-prev_buyers)/prev_buyers*100,1) as pct_buyers_change
from monthly

In [0]:
%sql
-- check: January vs October (+3.4%)
SELECT
  max(CASE WHEN month = '2019-10' THEN buyers END) AS oct_buyers,
  max(CASE WHEN month = '2020-01' THEN buyers END) AS jan_buyers,
  round(100 * (max(CASE WHEN month = '2020-01' THEN buyers END)
             - max(CASE WHEN month = '2019-10' THEN buyers END))
             / max(CASE WHEN month = '2019-10' THEN buyers END), 1) AS pct_jan_vs_oct
FROM dev.ecommerce.gold_overview

### Finding 1 · December was the peak, January went back to normal
- **December is the peak month:** 500,995 buyers. November grew fastest (+27.2% vs October).
- **January:** buyers fell **28.4%** vs December, but they were still **3.4% above October**.
- AOV moved the other way: December had the lowest AOV (354.87), January the highest (377.71). Fewer but bigger orders in January.
- **So:** most of the January drop is probably the end of the holiday season, not a business getting worse.
- **Careful:** the data has only one December-January, so I can't compare with last year.

In [0]:
%sql
SELECT * FROM dev.ecommerce.gold_funnel

In [0]:
%sql
with funnel as (
select 
month,
viewers,
carters,
buyers,
lag(viewers) over(order by month) as prev_viewers,
lag(carters) over(order by month) as prev_carters,
lag(buyers)  over(order by month) as prev_buyers from dev.ecommerce.gold_funnel
)

select 
month, 
viewers,
carters,
buyers,
round((viewers - prev_viewers)/prev_viewers * 100,1) as pct_viewers,
round((carters - prev_carters)/prev_carters * 100,1) as pct_carters,
round((buyers - prev_buyers)/prev_buyers * 100,1)   as pct_buyers from funnel



### Finding 2 · In January, people still came, but fewer of them bought
- From December to January: viewers **−4.2%**, carters **−19.8%**, buyers **−28.3%**.
- The drop gets bigger at each step of the funnel. Traffic stayed almost the same.
- Purchase rate (buyers / viewers): **10.9% → 8.2%**.
- **So:** the January loss is in **conversion, not traffic**. Probably the holiday shopping ended (see Finding 1), so the same visitors bought less.
- **Next question:** where in the funnel do we lose them, before or after the cart? → Finding 3.

In [0]:
%sql
SELECT * FROM dev.ecommerce.gold_cart_abandonment

### Finding 3 · Most of the January loss happens before the cart
- A cart = a session with at least one add-to-cart. Abandoned = the session has no purchase.
- Carts fell **22%** (1,983,487 → 1,543,484), but the abandoned rate rose only **2.5 points** (51.69% → 54.17%).
- **So:** people who add to the cart still buy at almost the same rate. The loss is **before the cart**: fewer people add products at all.
- Our abandoned rate (~52-54%) is well below the Baymard average (~70%).
- **Careful:** this is an upper bound. Someone who adds today and buys tomorrow in a new session counts as abandoned, so the real rate is probably lower.
- **Next question:** which products lose people before the cart? → Finding 4 (price).

In [0]:
%sql
SELECT * FROM dev.ecommerce.gold_funnel_price
ORDER BY price_band, month

In [0]:
%sql
-- January vs December: buyers change per price band
WITH bands AS (
  SELECT month,
         price_band,
         buyers,
         lag(buyers) OVER (PARTITION BY price_band ORDER BY month) AS prev_buyers
  FROM dev.ecommerce.gold_funnel_price
)
SELECT price_band,
       prev_buyers AS dec_buyers,
       buyers      AS jan_buyers,
       round((buyers - prev_buyers) / prev_buyers * 100, 1) AS pct_buyers_change
FROM bands
WHERE month = '2020-01'
ORDER BY price_band

### Finding 4 · Expensive products lose people before the cart, cheap ones after the cart
- **Before the cart:** the cart rate falls as the price goes up, from **14.5%** (<65) to **6.7%** (1000+) in December. People look at expensive products but don't add them.
- **After the cart:** the cheapest band (<65) has the lowest cart-to-purchase rate: **46.1%** in December, **39.5%** in January. The other bands are around 50-57%.
- Overall, the 65-165 band converts best (8.1% of viewers buy in December). The cheapest band is lower (6.7%) because it loses people after the cart.
- **January:** cheap bands lost the most buyers (<65 **−34.3%**), expensive bands the least (1000+ **−17.8%**). This explains why AOV went up in January (Finding 1): not bigger spending, but a **mix shift**, fewer cheap orders.
- **So:** for expensive products, the problem is the product page or the price. For cheap products, it is probably extra costs at checkout, like shipping (Baymard's top reason).
- **Careful:** the data has no shipping cost, so this is a hypothesis. It could be tested with an A/B test on a free-shipping threshold.

In [0]:
%sql
-- top 10 brands: December vs January purchase rate
SELECT 
brand,
max(CASE WHEN month = '2019-12' THEN viewers END)           AS dec_viewers,
max(CASE WHEN month = '2020-01' THEN viewers END)           AS jan_viewers,
max(CASE WHEN month = '2019-12' THEN purchase_rate_pct END) AS dec_rate,
max(CASE WHEN month = '2020-01' THEN purchase_rate_pct END) AS jan_rate
FROM dev.ecommerce.gold_funnel_brand
WHERE brand <> 'unknown'
GROUP BY brand
ORDER BY dec_viewers DESC
LIMIT 10

### Finding 5 · Big phone brands convert best, but January hit every brand
- `unknown` (brand missing, 1.41M viewers in December) is left out of the ranking.
- **Samsung and Apple convert the same:** 9.6% vs 9.7% in December, 7.7% vs 7.7% in January. Samsung is not better, just bigger (1.42M vs 0.98M viewers).
- The big phone brands (Samsung, Apple, Xiaomi, Huawei: 7.7-9.7%) convert better than the rest of the top 10 (3.4-6.9%).
- **January:** the purchase rate fell for **all** top 10 brands, so the drop is not a brand problem.
- **Apple** is the only top 10 brand with more viewers in January (980K → 1.01M), but its rate still fell (9.7% → 7.7%). More traffic, fewer buyers: the same story as Finding 2.
- **Careful:** brands sell different products (phones vs TVs vs appliances). The gap may come from the product type, not the brand. Category labels are not reliable, so I can't separate the two.

In [0]:
%sql
-- cohorts: who came back in each later month
SELECT * FROM dev.ecommerce.gold_retention
ORDER BY cohort_month, order_month

In [0]:
%sql
-- every month: new vs returning buyers
SELECT order_month,
sum(CASE WHEN cohort_month = order_month THEN buyers ELSE 0 END) AS new_buyers,
sum(CASE WHEN cohort_month < order_month THEN buyers ELSE 0 END) AS returning_buyers,
sum(buyers) AS all_buyers,
round(100 * sum(CASE WHEN cohort_month < order_month THEN buyers ELSE 0 END)
 / sum(buyers), 1) AS returning_pct
FROM dev.ecommerce.gold_retention
GROUP BY order_month
ORDER BY order_month

### Finding 6 · Returning customers stayed, the January drop came from new customers
- **Cohorts:** each new cohort came back less the next month: **26.3%** (Oct → Nov), **22.1%** (Nov → Dec), **15.2%** (Dec → Jan). The December cohort is probably full of one-time holiday shoppers, but its next month is January, which is weak for everyone. With one January only, I can't separate the two.
- **New vs returning, December → January:** new buyers **347,286 → 215,886** (−37.8%), returning buyers **153,709 → 142,982** (−7.0%). About **9 of every 10 lost buyers were new customers.**
- **So:** the loyal base held. The January drop is mostly fewer new customers after the holiday season.
- **Check:** new + returning = all buyers, and the totals match `gold_overview` for every month.
- **Careful:** the data starts in October 2019, so "new" means "first order since October". Some "new" customers may have bought before. For the same reason, the returning share (20.7% → 39.8%) grows partly because we can see more months back.

## Finding 7 · Answer and recommendation
**Main question:** where do we lose our customers, before they buy or after their first order?

**Answer: mostly before they buy, before the cart.**
- In January, traffic fell only **4%** (Finding 2), and carts fell **22%**, but the abandoned rate rose only **2.5 points** (Finding 3). People still come and look, but fewer of them add products to the cart.
- Not after the first order: returning buyers fell only **7%**, while new buyers fell **38%** (Finding 6).

**What I would do first:**
1. **Product page A/B tests, starting with expensive products.** The cart rate falls from 14.5% to 6.7% as the price goes up (Finding 4), so the biggest loss is on expensive products. Test simple changes there first: bigger photos, a clearer price, a simpler page. Then roll out what works to all products.
   - Primary metric: cart rate (carters / viewers).
   - Guardrail: purchase rate and revenue per visitor, so more carts do not come with fewer orders.
2. **Free-shipping threshold A/B test for cheap products.** The cheapest band loses the most people after the cart (cart-to-purchase 46.1% in December, 39.5% in January). Shipping cost is probably too high for a cheap product. Test a free-shipping threshold before changing it for everyone.
   - Primary metric: cart-to-purchase rate in the <65 band.
   - Guardrail: profit per order, because free shipping costs money.

**Why not something else first:**
- Not marketing: traffic barely fell (−4%), so more ads would bring people who still don't add to cart.
- Not retention: returning buyers held (−7%), so the loyal base is not the problem.

**Limits:** one December-January only (no last year to compare), no shipping cost or device data, category labels are not reliable, and "new" means "first order since October 2019".